# Final model and deployment export


In [1]:
import os
import json
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
import sklearn
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


## Feature set selection
Compare cross-validation performance from 06_language_features to select either the v1 or v2 dataset.


In [2]:
# cv results from 06_language_features
cv_mae_v1 = 27440.2956
cv_std_v1 = 163.4447
cv_mae_v2 = 26376.1372
cv_std_v2 = 114.1783

cv_improvement = cv_mae_v1 - cv_mae_v2

print(f"v1 CV MAE: ${cv_mae_v1:,.2f} +/- ${cv_std_v1:,.2f}")
print(f"v2 CV MAE: ${cv_mae_v2:,.2f} +/- ${cv_std_v2:,.2f}")
print(f"CV improvement (v1 - v2): ${cv_improvement:,.2f}")
print(f"Decision threshold (v1 CV std): ${cv_std_v1:,.2f}")

if cv_improvement > cv_std_v1:
    selected_dataset = 'cleaned_salary_data_v2.csv'
else:
    selected_dataset = 'cleaned_salary_data.csv'

print(f"Selected dataset: {selected_dataset}")


v1 CV MAE: $27,440.30 +/- $163.44
v2 CV MAE: $26,376.14 +/- $114.18
CV improvement (v1 - v2): $1,064.16
Decision threshold (v1 CV std): $163.44
Selected dataset: cleaned_salary_data_v2.csv


My observation: 


## Train/test split and preprocessor
Load the selected dataset, apply an 80/20 train/test split, and configure the column transformer.


In [3]:
data_path = os.path.join('..', 'data', selected_dataset)
df = pd.read_csv(data_path)

base_features = [
    'Country', 'Education', 'Role', 'RemoteWork', 'OrgSize',
    'Experience_Years', 'Work_Experience_Years'
]
lang_features = [c for c in df.columns if c not in base_features + ['Salary']]
feature_columns = base_features + lang_features

cat_cols = ['Country', 'Education', 'Role', 'RemoteWork', 'OrgSize']
num_cols = ['Experience_Years', 'Work_Experience_Years']

X = df[feature_columns].copy()
y = df['Salary'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols),
    ('lang', 'passthrough', lang_features)
])

print(f"Dataset shape: {df.shape}")
print(f"Training shape: {X_train.shape}, Test shape: {X_test.shape}")


Dataset shape: (15940, 23)
Training shape: (12752, 22), Test shape: (3188, 22)


My observation: 


## Quantile models training and evaluation
Fit three quantile HistGradientBoosting models for quantiles 0.25, 0.50, and 0.75 on log salary and evaluate test predictions.


In [4]:
quantiles = [0.25, 0.50, 0.75]
models = {}

for q in quantiles:
    pipe = Pipeline([
        ('prep', preprocessor),
        ('reg', HistGradientBoostingRegressor(loss='quantile', quantile=q, random_state=42))
    ])
    pipe.fit(X_train, np.log(y_train))
    models[q] = pipe

# raw test predictions
pred_raw_25 = np.exp(models[0.25].predict(X_test))
pred_raw_50 = np.exp(models[0.50].predict(X_test))
pred_raw_75 = np.exp(models[0.75].predict(X_test))

# check non-ascending predictions
non_ascending = ~((pred_raw_25 <= pred_raw_50) & (pred_raw_50 <= pred_raw_75))
non_ascending_count = int(np.sum(non_ascending))
non_ascending_share = float(np.mean(non_ascending) * 100)

print(f"Non-ascending test rows: {non_ascending_count} out of {len(y_test)} ({non_ascending_share:.2f}%)")

# sort predictions per row
pred_matrix = np.column_stack([pred_raw_25, pred_raw_50, pred_raw_75])
pred_sorted = np.sort(pred_matrix, axis=1)

pred_25 = pred_sorted[:, 0]
pred_50 = pred_sorted[:, 1]
pred_75 = pred_sorted[:, 2]

# test metrics for 0.50 model
mae_50 = mean_absolute_error(y_test, pred_50)
r2_50 = r2_score(y_test, pred_50)
print(f"Quantile 0.50 test MAE: ${mae_50:,.2f}")
print(f"Quantile 0.50 test R-squared: {r2_50:.4f}")

# overall coverage
in_interval = (y_test.values >= pred_25) & (y_test.values <= pred_75)
coverage_overall = float(np.mean(in_interval) * 100)
print(f"Overall test coverage (25%-75% interval): {coverage_overall:.2f}%")


Non-ascending test rows: 62 out of 3188 (1.94%)
Quantile 0.50 test MAE: $25,921.45
Quantile 0.50 test R-squared: 0.5976
Overall test coverage (25%-75% interval): 46.20%


My observation: 


## Coverage per country
Compute coverage and median prediction error across key focus countries on the test set.


In [5]:
focus_countries = [
    'India', 'Brazil', 'Poland', 'Germany',
    'United Kingdom of Great Britain and Northern Ireland',
    'United States of America'
]

test_df = X_test.copy()
test_df['actual'] = y_test.values
test_df['pred_25'] = pred_25
test_df['pred_50'] = pred_50
test_df['pred_75'] = pred_75
test_df['in_interval'] = in_interval

country_rows = []
for c in focus_countries:
    sub = test_df[test_df['Country'] == c]
    cov = sub['in_interval'].mean() * 100
    mae_c = mean_absolute_error(sub['actual'], sub['pred_50'])
    country_rows.append({
        'Country': c,
        'Test row count': len(sub),
        'Coverage (%)': cov,
        'MAE ($)': mae_c
    })

df_country_cov = pd.DataFrame(country_rows)
display(df_country_cov)


,Country,Test row count,Coverage (%),MAE ($)
0,India,91,51.6484,"17,648.1711"
1,Brazil,65,52.3077,"17,308.5382"
2,Poland,65,47.6923,"19,373.6585"
3,Germany,344,46.2209,"19,915.9158"
4,United Kingdom of Great Britain and Northern I...,227,41.8502,"30,141.2029"
5,United States of America,751,49.0013,"38,079.4744"


My observation: 


## Comparison with squared-error HistGB
Compare test performance of the median quantile model against the squared-error HistGB model.


In [6]:
pipe_squared = Pipeline([
    ('prep', preprocessor),
    ('reg', HistGradientBoostingRegressor(loss='squared_error', random_state=42))
])
pipe_squared.fit(X_train, np.log(y_train))
pred_squared = np.exp(pipe_squared.predict(X_test))

comparison_table = pd.DataFrame([
    {
        'Model': 'Squared-error HistGB (v2 features)',
        'Test MAE ($)': mean_absolute_error(y_test, pred_squared),
        'Test R-squared': r2_score(y_test, pred_squared)
    },
    {
        'Model': 'Quantile 0.50 HistGB (v2 features)',
        'Test MAE ($)': mae_50,
        'Test R-squared': r2_50
    }
])
display(comparison_table)


,Model,Test MAE ($),Test R-squared
0,Squared-error HistGB (v2 features),"25,858.0652",0.6013
1,Quantile 0.50 HistGB (v2 features),"25,921.4530",0.5976


My observation: 


## Full dataset refit and export
Refit the quantile pipelines on all cleaned rows and save the serialized model and schema to the ml/artifacts directory.


In [7]:
full_models = {}
for q in quantiles:
    pipe_full = Pipeline([
        ('prep', preprocessor),
        ('reg', HistGradientBoostingRegressor(loss='quantile', quantile=q, random_state=42))
    ])
    pipe_full.fit(X, np.log(y))
    full_models[f'q{int(q*100)}'] = pipe_full

print(f"Refit all 3 quantile models on all {len(X)} cleaned rows.")

artifacts_dir = os.path.join('..', 'ml', 'artifacts')
os.makedirs(artifacts_dir, exist_ok=True)

model_artifact = {
    'models': full_models,
    'feature_columns': feature_columns,
    'categorical_columns': cat_cols,
    'numeric_columns': num_cols,
    'sklearn_version': sklearn.__version__,
    'trained_on_rows': len(X),
    'test_metrics': {
        'mae_50': float(mae_50),
        'r2_50': float(r2_50),
        'coverage_overall': float(coverage_overall),
        'non_ascending_share': float(non_ascending_share)
    },
    'created_at': datetime.now().isoformat()
}

joblib_path = os.path.join(artifacts_dir, 'salary_model.joblib')
joblib.dump(model_artifact, joblib_path)
print(f"Saved model artifact to: {joblib_path}")

schema = {
    'categorical': {
        col: sorted([str(x) for x in X[col].unique().tolist()])
        for col in cat_cols
    },
    'numeric': {
        col: {
            'min': float(X[col].min()),
            'max': float(X[col].max())
        }
        for col in num_cols
    },
    'languages': {
        col: {'min': 0, 'max': 1}
        for col in lang_features
    }
}

schema_path = os.path.join(artifacts_dir, 'schema.json')
with open(schema_path, 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2)

print(f"Saved schema artifact to: {schema_path}")


Refit all 3 quantile models on all 15940 cleaned rows.
Saved model artifact to: ..\ml\artifacts\salary_model.joblib
Saved schema artifact to: ..\ml\artifacts\schema.json


My observation: 
